In [1]:
from manim import *
import numpy as np
from scipy.integrate import odeint
from scipy.interpolate import interp1d

class MassaMola(Scene):
    def construct(self):
        # 1. Simulação Numérica
        m, c, k, F = 1.0, 0.5, 2.0, 1.0
        
        def sistema(y, t): 
            return [y[1], (F - c*y[1] - k*y[0])/m]
        
        t_max = 15
        t_arr = np.linspace(0, t_max, 300)
        sol = odeint(sistema, [0.0, 0.0], t_arr)
        
        # Interpolação para garantir fluidez em qualquer FPS do Manim
        pos_func = interp1d(t_arr, sol[:, 0], kind='cubic')

        # 2. Elementos da Cena
        tracker_t = ValueTracker(0)

        # Gráfico (Direita)
        eixos = Axes(
            x_range=[0, t_max, 3], y_range=[-0.2, 1.2, 0.5],
            x_length=6, y_length=4
        ).to_edge(RIGHT)
        labels = eixos.get_axis_labels(x_label="t", y_label="x")
        
        # Sistema Físico (Esquerda)
        parede = Line(UP*1.5, DOWN*1.5).shift(LEFT*6)
        parede.set_stroke(width=6)
        
        massa = Square(side_length=1.2, color=BLUE, fill_opacity=0.8)
        
        # 3. Funções de Atualização (Updaters)
        def update_massa(mob):
            x = pos_func(tracker_t.get_value())
            # x=-3 é a posição de repouso visual; fator 2 amplifica o movimento na tela
            mob.move_to(LEFT*3 + RIGHT * float(x) * 2)

        massa.add_updater(update_massa)

        def get_mola():
            start = parede.get_center() + UP*0.3
            end = massa.get_left() + UP*0.3
            L = np.linalg.norm(end - start)
            pts = [start]
            for i in range(1, 12):
                x_p = start[0] + (L/12)*i
                y_p = start[1] + (0.15 if i%2==0 else -0.15)
                pts.append([x_p, y_p, 0])
            pts.append(end)
            return VMobject(color=WHITE).set_points_as_corners(pts)

        def get_amortecedor():
            start = parede.get_center() + DOWN*0.3
            end = massa.get_left() + DOWN*0.3
            meio = start[0] + 0.8 # Comprimento fixo do cilindro
            cilindro = Line(start, [meio, start[1], 0], color=GREY, stroke_width=8)
            pistao = Line([meio - 0.3, end[1], 0], end, color=WHITE, stroke_width=4)
            return VGroup(cilindro, pistao)

        # Redesenha a mola e o amortecedor a cada frame
        mola = always_redraw(get_mola)
        amortecedor = always_redraw(get_amortecedor)

        # Ponto no gráfico e rastro
        ponto = always_redraw(lambda: Dot(
            eixos.c2p(tracker_t.get_value(), pos_func(tracker_t.get_value())),
            color=RED
        ))
        rastro = TracedPath(ponto.get_center, stroke_color=RED, stroke_width=4)

        # 4. Renderização
        self.add(eixos, labels, parede, massa, mola, amortecedor, ponto, rastro)
        
        # Executa a animação manipulando o tempo linearmente
        self.play(tracker_t.animate.set_value(t_max), run_time=8, rate_func=linear)
        self.wait(1)